In [7]:
import os 
import sys
sys.path.append('/home/hb-nano/mirsaid/face-recognition')
from ultralytics import YOLO
import cv2

data_images = '/home/hb-nano/mirsaid/face-recognition/data/ilhan'
model = YOLO('/home/hb-nano/mirsaid/face-recognition/yolov8m-face.pt')

for filename in os.listdir(data_images):
    image = cv2.imread(os.path.join(data_images, filename))

    # Detect and crop face
    yolo_results = model(image, max_det=1, verbose=False)

    x1, y1, x2, y2, _, _ = yolo_results[0].boxes.data[0].cpu().numpy()

    face_image = image[int(y1):int(y2), int(x1):int(x2)]

    # save image to the folder
    cv2.imwrite(f'/home/hb-nano/mirsaid/face-recognition/data/ilhan_2/{filename}.jpg', face_image)

In [3]:
import cv2
import pandas as pd
import random

video_names = ['videoa1-5']

for video_name in video_names:
    gt_path = f'/home/hbvision/mirsaid/smart-office/TrackEval/data/gt/mot_challenge/ilhan-train/{video_name}/gt/gt.txt'
    video_path = f'/home/hbvision/mirsaid/smart-office/client/pred_videos/{video_name}_eval.mp4'

    # Load ground truth data
    gt_data = pd.read_csv(gt_path, header=None)
    gt_data.columns = ['frame', 'id', 'x', 'y', 'w', 'h', 'conf', 'class', 'visibility']

    # Sort by frame number and filter visible objects
    gt_data = gt_data.sort_values(by='frame')

    # Assign a random color to each object ID
    id_colors = {}
    unique_ids = gt_data['id'].unique()
    for obj_id in unique_ids:
        id_colors[obj_id] = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))


    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    out = cv2.VideoWriter(f'output_{video_name}.avi', cv2.VideoWriter_fourcc(*'XVID'), 30, (width, height))

    frame_num = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_num += 1
        frame_data = gt_data[gt_data['frame'] == frame_num]

        for _, row in frame_data.iterrows():
            if row['class'] != 1:
                continue

            x, y, w, h = int(row['x']), int(row['y']), int(row['w']), int(row['h'])
            obj_id = int(row['id'])
            visibility = row['visibility']

            # Get color for the current object ID
            color = id_colors[obj_id]

            # Draw bounding box
            cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)

            # Add text for ID and visibility level
            text = f'ID: {obj_id}, Vis: {visibility:.2f}'
            cv2.putText(frame, text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        out.write(frame)
        cv2.imshow('Tracking', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Cleanup
    out.release()
    cap.release()
    cv2.destroyAllWindows()
    print(f'Video {video_name} processed', end='')

Video videoa1-5 processed

In [5]:
import numpy as np
import pandas as pd

gt_path = f'/home/hbvision/mirsaid/smart-office/TrackEval/data/gt/mot_challenge/hbface-train/video10/gt/gt.txt'
pred_path = f'/home/hbvision/mirsaid/smart-office/results/video10_recognition_alg1.txt'

# Get unique track IDs from ground truth
gt_df = pd.read_csv(gt_path, names=['frame', 'track_id', 'x', 'y', 'w', 'h', 'confidence', 'class', 'visibility'])
gt_df = gt_df[['frame', 'track_id']]
gt_df = gt_df.drop_duplicates(subset=['track_id'])

gt_ids = set(gt_df['track_id'])
gt_ids_aranged = np.arange(1, len(gt_ids) + 1)

# Get unique track IDs from predictions
pred_df = pd.read_csv(pred_path, names=['frame', 'x', 'y', 'w', 'h', 'name'])
unique_names = pred_df['name'].unique()
pred_ids_aranged = np.arange(1, len(unique_names) + 1)

# Calculate accuracy with aranged IDs
correct = len(set(gt_ids_aranged) & set(pred_ids_aranged))
total = len(gt_ids_aranged)

accuracy = correct / total if total > 0 else 0

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.7500
